# C / C++ 基本指標 50 題：自動檢查版

本題本依照你提供的 C / C++ 自動檢查 Notebook 版型製作，主題集中在 **基本指標**。

## 作答流程

1. 先執行下面的「C / C++ 自動檢查工具」cell。
2. 每題都有兩個作答 cell：
   - `%%c_answer Q...`：C（C11）
   - `%%cpp_answer Q...`：C++（C++17）
3. **不需要在作答 cell 手動輸入測試資料。**
4. 完成某一語言後，執行緊接在該作答 cell 下方的檢查。
5. 每題使用 **3 組測試資料**。
6. 最後執行「總檢查」查看完成度。

> 可以只練 C、只練 C++，或同一題兩種都寫。題目由基礎逐步增加整合程度。


In [1]:
# === C / C++ 自動檢查工具：請先執行這個 cell，不要修改 ===
import base64
import json
import os
import shutil
import subprocess
import tempfile
from IPython.core.magic import register_cell_magic

_TEST_CASES = json.loads(
    base64.b64decode("eyJRMSI6IFt7ImlucHV0IjogIjhcbiIsICJleHBlY3RlZCI6ICI4XG4ifSwgeyJpbnB1dCI6ICItM1xuIiwgImV4cGVjdGVkIjogIi0zXG4ifSwgeyJpbnB1dCI6ICIwXG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn1dLCAiUTIiOiBbeyJpbnB1dCI6ICIxMFxuIiwgImV4cGVjdGVkIjogIjE1XG4ifSwgeyJpbnB1dCI6ICItNVxuIiwgImV4cGVjdGVkIjogIjBcbiJ9LCB7ImlucHV0IjogIjBcbiIsICJleHBlY3RlZCI6ICI1XG4ifV0sICJRMyI6IFt7ImlucHV0IjogIjdcbiIsICJleHBlY3RlZCI6ICIxNFxuIn0sIHsiaW5wdXQiOiAiLTRcbiIsICJleHBlY3RlZCI6ICItOFxuIn0sIHsiaW5wdXQiOiAiMjVcbiIsICJleHBlY3RlZCI6ICI1MFxuIn1dLCAiUTQiOiBbeyJpbnB1dCI6ICIxMlxuIiwgImV4cGVjdGVkIjogIi0xMlxuIn0sIHsiaW5wdXQiOiAiLTlcbiIsICJleHBlY3RlZCI6ICI5XG4ifSwgeyJpbnB1dCI6ICIwXG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn1dLCAiUTUiOiBbeyJpbnB1dCI6ICIzIDdcbiIsICJleHBlY3RlZCI6ICIxMFxuIn0sIHsiaW5wdXQiOiAiLTUgMlxuIiwgImV4cGVjdGVkIjogIi0zXG4ifSwgeyJpbnB1dCI6ICIxMDAgMjBcbiIsICJleHBlY3RlZCI6ICIxMjBcbiJ9XSwgIlE2IjogW3siaW5wdXQiOiAiOSA0XG4iLCAiZXhwZWN0ZWQiOiAiNVxuIn0sIHsiaW5wdXQiOiAiMiA4XG4iLCAiZXhwZWN0ZWQiOiAiLTZcbiJ9LCB7ImlucHV0IjogIi0zIC03XG4iLCAiZXhwZWN0ZWQiOiAiNFxuIn1dLCAiUTciOiBbeyJpbnB1dCI6ICI1XG4iLCAiZXhwZWN0ZWQiOiAiMjVcbiJ9LCB7ImlucHV0IjogIi02XG4iLCAiZXhwZWN0ZWQiOiAiMzZcbiJ9LCB7ImlucHV0IjogIjBcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJROCI6IFt7ImlucHV0IjogIkFcbiIsICJleHBlY3RlZCI6ICJBXG4ifSwgeyJpbnB1dCI6ICJ6XG4iLCAiZXhwZWN0ZWQiOiAielxuIn0sIHsiaW5wdXQiOiAiN1xuIiwgImV4cGVjdGVkIjogIjdcbiJ9XSwgIlE5IjogW3siaW5wdXQiOiAiYVxuIiwgImV4cGVjdGVkIjogIkFcbiJ9LCB7ImlucHV0IjogIm1cbiIsICJleHBlY3RlZCI6ICJNXG4ifSwgeyJpbnB1dCI6ICJ6XG4iLCAiZXhwZWN0ZWQiOiAiWlxuIn1dLCAiUTEwIjogW3siaW5wdXQiOiAiMFxuIiwgImV4cGVjdGVkIjogIk5VTExcbiJ9LCB7ImlucHV0IjogIjlcbiIsICJleHBlY3RlZCI6ICI5XG4ifSwgeyJpbnB1dCI6ICItNFxuIiwgImV4cGVjdGVkIjogIi00XG4ifV0sICJRMTEiOiBbeyJpbnB1dCI6ICI5XG4iLCAiZXhwZWN0ZWQiOiAiMTBcbiJ9LCB7ImlucHV0IjogIi0xXG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn0sIHsiaW5wdXQiOiAiMTAwXG4iLCAiZXhwZWN0ZWQiOiAiMTAxXG4ifV0sICJRMTIiOiBbeyJpbnB1dCI6ICIzNVxuIiwgImV4cGVjdGVkIjogIjBcbiJ9LCB7ImlucHV0IjogIi0xMlxuIiwgImV4cGVjdGVkIjogIjBcbiJ9LCB7ImlucHV0IjogIjBcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRMTMiOiBbeyJpbnB1dCI6ICI0XG4iLCAiZXhwZWN0ZWQiOiAiMTJcbiJ9LCB7ImlucHV0IjogIi0zXG4iLCAiZXhwZWN0ZWQiOiAiLTlcbiJ9LCB7ImlucHV0IjogIjBcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRMTQiOiBbeyJpbnB1dCI6ICIzIDhcbiIsICJleHBlY3RlZCI6ICI4IDNcbiJ9LCB7ImlucHV0IjogIi0xIDlcbiIsICJleHBlY3RlZCI6ICI5IC0xXG4ifSwgeyJpbnB1dCI6ICI1IDVcbiIsICJleHBlY3RlZCI6ICI1IDVcbiJ9XSwgIlExNSI6IFt7ImlucHV0IjogIjkgMlxuIiwgImV4cGVjdGVkIjogIjIgOVxuIn0sIHsiaW5wdXQiOiAiMyA3XG4iLCAiZXhwZWN0ZWQiOiAiMyA3XG4ifSwgeyJpbnB1dCI6ICItMiAtOFxuIiwgImV4cGVjdGVkIjogIi04IC0yXG4ifV0sICJRMTYiOiBbeyJpbnB1dCI6ICIxNyA1XG4iLCAiZXhwZWN0ZWQiOiAiMyAyXG4ifSwgeyJpbnB1dCI6ICIyMCA0XG4iLCAiZXhwZWN0ZWQiOiAiNSAwXG4ifSwgeyJpbnB1dCI6ICItMTcgNVxuIiwgImV4cGVjdGVkIjogIi0zIC0yXG4ifV0sICJRMTciOiBbeyJpbnB1dCI6ICI4IDNcbiIsICJleHBlY3RlZCI6ICIzIDhcbiJ9LCB7ImlucHV0IjogIi00IC05XG4iLCAiZXhwZWN0ZWQiOiAiLTkgLTRcbiJ9LCB7ImlucHV0IjogIjUgNVxuIiwgImV4cGVjdGVkIjogIjUgNVxuIn1dLCAiUTE4IjogW3siaW5wdXQiOiAiLTEyXG4iLCAiZXhwZWN0ZWQiOiAiMTJcbiJ9LCB7ImlucHV0IjogIjdcbiIsICJleHBlY3RlZCI6ICI3XG4ifSwgeyJpbnB1dCI6ICIwXG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn1dLCAiUTE5IjogW3siaW5wdXQiOiAiQVxuIiwgImV4cGVjdGVkIjogImFcbiJ9LCB7ImlucHV0IjogIk1cbiIsICJleHBlY3RlZCI6ICJtXG4ifSwgeyJpbnB1dCI6ICJaXG4iLCAiZXhwZWN0ZWQiOiAielxuIn1dLCAiUTIwIjogW3siaW5wdXQiOiAiMiA1IDNcbiIsICJleHBlY3RlZCI6ICI1IDhcbiJ9LCB7ImlucHV0IjogIi0xIDEwIDRcbiIsICJleHBlY3RlZCI6ICIzIDE0XG4ifSwgeyJpbnB1dCI6ICIwIDAgLTJcbiIsICJleHBlY3RlZCI6ICItMiAtMlxuIn1dLCAiUTIxIjogW3siaW5wdXQiOiAiNFxuIiwgImV4cGVjdGVkIjogIjE4XG4ifSwgeyJpbnB1dCI6ICIwXG4iLCAiZXhwZWN0ZWQiOiAiNlxuIn0sIHsiaW5wdXQiOiAiLTJcbiIsICJleHBlY3RlZCI6ICIwXG4ifV0sICJRMjIiOiBbeyJpbnB1dCI6ICI1XG4iLCAiZXhwZWN0ZWQiOiAiMTVcbiJ9LCB7ImlucHV0IjogIi0yMFxuIiwgImV4cGVjdGVkIjogIi0xMFxuIn0sIHsiaW5wdXQiOiAiMFxuIiwgImV4cGVjdGVkIjogIjEwXG4ifV0sICJRMjMiOiBbeyJpbnB1dCI6ICIzIDggMFxuIiwgImV4cGVjdGVkIjogIjNcbiJ9LCB7ImlucHV0IjogIjMgOCAxXG4iLCAiZXhwZWN0ZWQiOiAiOFxuIn0sIHsiaW5wdXQiOiAiLTUgNyA5XG4iLCAiZXhwZWN0ZWQiOiAiN1xuIn1dLCAiUTI0IjogW3siaW5wdXQiOiAiMyA5XG4iLCAiZXhwZWN0ZWQiOiAiOVxuIn0sIHsiaW5wdXQiOiAiOCAyXG4iLCAiZXhwZWN0ZWQiOiAiOFxuIn0sIHsiaW5wdXQiOiAiNSA1XG4iLCAiZXhwZWN0ZWQiOiAiNVxuIn1dLCAiUTI1IjogW3siaW5wdXQiOiAiMyA5XG4iLCAiZXhwZWN0ZWQiOiAiMTAzIDlcbiJ9LCB7ImlucHV0IjogIjggMlxuIiwgImV4cGVjdGVkIjogIjggMTAyXG4ifSwgeyJpbnB1dCI6ICI1IDVcbiIsICJleHBlY3RlZCI6ICIxMDUgNVxuIn1dLCAiUTI2IjogW3siaW5wdXQiOiAiNlxuIiwgImV4cGVjdGVkIjogIjEyXG4ifSwgeyJpbnB1dCI6ICItNFxuIiwgImV4cGVjdGVkIjogIi04XG4ifSwgeyJpbnB1dCI6ICIwXG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn1dLCAiUTI3IjogW3siaW5wdXQiOiAiMVxuIiwgImV4cGVjdGVkIjogIjhcbiJ9LCB7ImlucHV0IjogIi03XG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn0sIHsiaW5wdXQiOiAiMTAwXG4iLCAiZXhwZWN0ZWQiOiAiMTA3XG4ifV0sICJRMjgiOiBbeyJpbnB1dCI6ICI0IDExXG4iLCAiZXhwZWN0ZWQiOiAiMTFcbiJ9LCB7ImlucHV0IjogIjkgMlxuIiwgImV4cGVjdGVkIjogIjlcbiJ9LCB7ImlucHV0IjogIi0zIC0zXG4iLCAiZXhwZWN0ZWQiOiAiLTNcbiJ9XSwgIlEyOSI6IFt7ImlucHV0IjogIjQgMTFcbiIsICJleHBlY3RlZCI6ICI0XG4ifSwgeyJpbnB1dCI6ICI5IDJcbiIsICJleHBlY3RlZCI6ICIyXG4ifSwgeyJpbnB1dCI6ICItMyAtOFxuIiwgImV4cGVjdGVkIjogIi04XG4ifV0sICJRMzAiOiBbeyJpbnB1dCI6ICIxMCAyMFxuIiwgImV4cGVjdGVkIjogIjMwXG4ifSwgeyJpbnB1dCI6ICItNSA4XG4iLCAiZXhwZWN0ZWQiOiAiM1xuIn0sIHsiaW5wdXQiOiAiMCAwXG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn1dLCAiUTMxIjogW3siaW5wdXQiOiAiOFxuIiwgImV4cGVjdGVkIjogIjlcbiJ9LCB7ImlucHV0IjogIi0yXG4iLCAiZXhwZWN0ZWQiOiAiLTFcbiJ9LCB7ImlucHV0IjogIjBcbiIsICJleHBlY3RlZCI6ICIxXG4ifV0sICJRMzIiOiBbeyJpbnB1dCI6ICIxNFxuIiwgImV4cGVjdGVkIjogIjE0XG4ifSwgeyJpbnB1dCI6ICItNlxuIiwgImV4cGVjdGVkIjogIi02XG4ifSwgeyJpbnB1dCI6ICIwXG4iLCAiZXhwZWN0ZWQiOiAiMFxuIn1dLCAiUTMzIjogW3siaW5wdXQiOiAiOTlcbiIsICJleHBlY3RlZCI6ICI5OVxuIn0sIHsiaW5wdXQiOiAiLTFcbiIsICJleHBlY3RlZCI6ICItMVxuIn0sIHsiaW5wdXQiOiAiN1xuIiwgImV4cGVjdGVkIjogIjdcbiJ9XSwgIlEzNCI6IFt7ImlucHV0IjogIjEuNVxuIiwgImV4cGVjdGVkIjogIjMuMDBcbiJ9LCB7ImlucHV0IjogIi0yLjI1XG4iLCAiZXhwZWN0ZWQiOiAiLTQuNTBcbiJ9LCB7ImlucHV0IjogIjBcbiIsICJleHBlY3RlZCI6ICIwLjAwXG4ifV0sICJRMzUiOiBbeyJpbnB1dCI6ICIxIDQyXG4iLCAiZXhwZWN0ZWQiOiAiNDJcbiJ9LCB7ImlucHV0IjogIjAgNDJcbiIsICJleHBlY3RlZCI6ICJOVUxMXG4ifSwgeyJpbnB1dCI6ICIxIC0zXG4iLCAiZXhwZWN0ZWQiOiAiLTNcbiJ9XSwgIlEzNiI6IFt7ImlucHV0IjogIjEgMlxuIiwgImV4cGVjdGVkIjogIjIgMVxuIn0sIHsiaW5wdXQiOiAiMTAgLTVcbiIsICJleHBlY3RlZCI6ICItNSAxMFxuIn0sIHsiaW5wdXQiOiAiNyA3XG4iLCAiZXhwZWN0ZWQiOiAiNyA3XG4ifV0sICJRMzciOiBbeyJpbnB1dCI6ICI1XG4iLCAiZXhwZWN0ZWQiOiAiNlxuIn0sIHsiaW5wdXQiOiAiLTFcbiIsICJleHBlY3RlZCI6ICIwXG4ifSwgeyJpbnB1dCI6ICIwXG4iLCAiZXhwZWN0ZWQiOiAiMVxuIn1dLCAiUTM4IjogW3siaW5wdXQiOiAiMyA0XG4iLCAiZXhwZWN0ZWQiOiAiMyA4XG4ifSwgeyJpbnB1dCI6ICItMiA1XG4iLCAiZXhwZWN0ZWQiOiAiLTIgMTBcbiJ9LCB7ImlucHV0IjogIjcgMFxuIiwgImV4cGVjdGVkIjogIjcgMFxuIn1dLCAiUTM5IjogW3siaW5wdXQiOiAiYlxuIiwgImV4cGVjdGVkIjogIkJcbiJ9LCB7ImlucHV0IjogIlFcbiIsICJleHBlY3RlZCI6ICJRXG4ifSwgeyJpbnB1dCI6ICJ6XG4iLCAiZXhwZWN0ZWQiOiAiWlxuIn1dLCAiUTQwIjogW3siaW5wdXQiOiAiMTBcbiIsICJleHBlY3RlZCI6ICIxMlxuIn0sIHsiaW5wdXQiOiAiMFxuIiwgImV4cGVjdGVkIjogIjJcbiJ9LCB7ImlucHV0IjogIi01XG4iLCAiZXhwZWN0ZWQiOiAiLTNcbiJ9XSwgIlE0MSI6IFt7ImlucHV0IjogIi04IDNcbiIsICJleHBlY3RlZCI6ICIzIDhcbiJ9LCB7ImlucHV0IjogIi0yIC05XG4iLCAiZXhwZWN0ZWQiOiAiMiA5XG4ifSwgeyJpbnB1dCI6ICI1IDFcbiIsICJleHBlY3RlZCI6ICIxIDVcbiJ9XSwgIlE0MiI6IFt7ImlucHV0IjogIjMgNCAyXG4iLCAiZXhwZWN0ZWQiOiAiMTRcbiJ9LCB7ImlucHV0IjogIi0yIDUgMVxuIiwgImV4cGVjdGVkIjogIi05XG4ifSwgeyJpbnB1dCI6ICIxMCAwIDdcbiIsICJleHBlY3RlZCI6ICI3XG4ifV0sICJRNDMiOiBbeyJpbnB1dCI6ICIxMCAzXG4iLCAiZXhwZWN0ZWQiOiAiN1xuIn0sIHsiaW5wdXQiOiAiLTQgNVxuIiwgImV4cGVjdGVkIjogIjlcbiJ9LCB7ImlucHV0IjogIjcgN1xuIiwgImV4cGVjdGVkIjogIjBcbiJ9XSwgIlE0NCI6IFt7ImlucHV0IjogIjIgOCA1XG4iLCAiZXhwZWN0ZWQiOiAiNyA4XG4ifSwgeyJpbnB1dCI6ICI5IDMgNFxuIiwgImV4cGVjdGVkIjogIjkgN1xuIn0sIHsiaW5wdXQiOiAiNSA1IDEwXG4iLCAiZXhwZWN0ZWQiOiAiMTUgNVxuIn1dLCAiUTQ1IjogW3siaW5wdXQiOiAiMiA4IDNcbiIsICJleHBlY3RlZCI6ICIyIDVcbiJ9LCB7ImlucHV0IjogIjkgMyA0XG4iLCAiZXhwZWN0ZWQiOiAiNSAzXG4ifSwgeyJpbnB1dCI6ICI1IDUgMlxuIiwgImV4cGVjdGVkIjogIjMgNVxuIn1dLCAiUTQ2IjogW3siaW5wdXQiOiAiOCAyXG4iLCAiZXhwZWN0ZWQiOiAiMTAgNiAxNiA0XG4ifSwgeyJpbnB1dCI6ICI3IDNcbiIsICJleHBlY3RlZCI6ICIxMCA0IDIxIDJcbiJ9LCB7ImlucHV0IjogIi05IDJcbiIsICJleHBlY3RlZCI6ICItNyAtMTEgLTE4IC00XG4ifV0sICJRNDciOiBbeyJpbnB1dCI6ICIxNSAwIDEwXG4iLCAiZXhwZWN0ZWQiOiAiMTBcbiJ9LCB7ImlucHV0IjogIi00IDAgMTBcbiIsICJleHBlY3RlZCI6ICIwXG4ifSwgeyJpbnB1dCI6ICI2IDAgMTBcbiIsICJleHBlY3RlZCI6ICI2XG4ifV0sICJRNDgiOiBbeyJpbnB1dCI6ICIxIDkgMVxuIiwgImV4cGVjdGVkIjogIjkgMVxuIn0sIHsiaW5wdXQiOiAiMSA5IDBcbiIsICJleHBlY3RlZCI6ICIxIDlcbiJ9LCB7ImlucHV0IjogIi0zIDQgMVxuIiwgImV4cGVjdGVkIjogIjQgLTNcbiJ9XSwgIlE0OSI6IFt7ImlucHV0IjogIjNcbiIsICJleHBlY3RlZCI6ICIxNlxuIn0sIHsiaW5wdXQiOiAiLTVcbiIsICJleHBlY3RlZCI6ICIwXG4ifSwgeyJpbnB1dCI6ICIxMFxuIiwgImV4cGVjdGVkIjogIjMwXG4ifV0sICJRNTAiOiBbeyJpbnB1dCI6ICIzIDhcbiIsICJleHBlY3RlZCI6ICI3IDRcbiJ9LCB7ImlucHV0IjogIjAgMFxuIiwgImV4cGVjdGVkIjogIi0xIDFcbiJ9LCB7ImlucHV0IjogIi00IDZcbiIsICJleHBlY3RlZCI6ICI1IC0zXG4ifV19").decode("utf-8")
)

_ANSWERS = {"C": {}, "CPP": {}}
_RESULTS = {"C": {}, "CPP": {}}

def _normalise_output(text):
    lines = [line.rstrip() for line in text.replace("\r\n", "\n").split("\n")]
    while lines and lines[-1] == "":
        lines.pop()
    return "\n".join(lines)

def _compile_and_run(source, input_text, language):
    if language == "C":
        compiler = shutil.which("gcc")
        suffix = ".c"
        flags = ["-std=c11", "-O2"]
    else:
        compiler = shutil.which("g++")
        suffix = ".cpp"
        flags = ["-std=c++17", "-O2"]

    if compiler is None:
        return False, "", f"找不到 {'gcc' if language == 'C' else 'g++'}。請先安裝 C/C++ compiler。"

    with tempfile.TemporaryDirectory() as td:
        src = os.path.join(td, "answer" + suffix)
        exe = os.path.join(td, "answer")
        with open(src, "w", encoding="utf-8") as f:
            f.write(source)

        compile_result = subprocess.run(
            [compiler, src, "-o", exe] + flags,
            capture_output=True,
            text=True,
        )
        if compile_result.returncode != 0:
            return False, "", compile_result.stderr

        try:
            run_result = subprocess.run(
                [exe],
                input=input_text,
                capture_output=True,
                text=True,
                timeout=5,
            )
        except subprocess.TimeoutExpired:
            return False, "", "程式執行超過 5 秒，請檢查是否有無限迴圈。"

        if run_result.returncode != 0:
            err = run_result.stderr or f"程式結束碼：{run_result.returncode}"
            return False, run_result.stdout, err
        return True, run_result.stdout, ""

@register_cell_magic
def c_answer(line, cell):
    qid = line.strip().upper()
    _ANSWERS["C"][qid] = cell
    print(f"已儲存 {qid} 的 C 作答。請執行下方 C 檢查 cell。")

@register_cell_magic
def cpp_answer(line, cell):
    qid = line.strip().upper()
    _ANSWERS["CPP"][qid] = cell
    print(f"已儲存 {qid} 的 C++ 作答。請執行下方 C++ 檢查 cell。")

def _run_check(qid, language):
    qid = qid.upper()
    language = language.upper()
    label = "C" if language == "C" else "C++"

    if qid not in _ANSWERS[language]:
        print(f"❌ {qid}：尚未執行 {label} 作答 cell")
        _RESULTS[language][qid] = False
        return

    source = _ANSWERS[language][qid]
    cases = _TEST_CASES[qid]
    passed = 0

    for case_number, case in enumerate(cases, start=1):
        ok_run, stdout, error = _compile_and_run(source, case["input"], language)
        if not ok_run:
            print(f"❌ {qid}（{label}）：編譯或執行失敗")
            print(error)
            _RESULTS[language][qid] = False
            return

        actual = _normalise_output(stdout)
        expected = _normalise_output(case["expected"])

        if actual == expected:
            passed += 1
        else:
            print(f"❌ {qid}（{label}）測試 {case_number} 未通過")
            print("測試輸入：")
            print(case["input"], end="")
            print("你的輸出：")
            print(stdout if stdout else "<沒有輸出>")
            print("預期輸出：")
            print(case["expected"], end="")
            print()

    success = passed == len(cases)
    _RESULTS[language][qid] = success
    if success:
        print(f"✅ {qid}（{label}）全部通過（{passed} / {len(cases)}）")
    else:
        print(f"❌ {qid}（{label}）：通過 {passed} / {len(cases)} 個測試")

def _language_report(language):
    label = "C" if language == "C" else "C++"
    results = _RESULTS[language]
    passed = sum(results.get(f"Q{i}", False) for i in range(1, 51))
    untouched = [f"Q{i}" for i in range(1, 51) if f"Q{i}" not in results]
    print(f"{label}：{passed} / 50 題通過")
    if untouched:
        print("尚未執行檢查：" + ", ".join(untouched))

def _final_report():
    print("=== C / C++ 總檢查 ===")
    _language_report("C")
    print()
    _language_report("CPP")
    both_passed = sum(
        _RESULTS["C"].get(f"Q{i}", False)
        and _RESULTS["CPP"].get(f"Q{i}", False)
        for i in range(1, 51)
    )
    print()
    print(f"C 與 C++ 都通過：{both_passed} / 50 題")

print("C / C++ 自動檢查工具已載入。")
print("C 使用 C11；C++ 使用 C++17。")
print("每題有 3 組測試資料；作答後不需要手動輸入測試資料。")


C / C++ 自動檢查工具已載入。
C 使用 C11；C++ 使用 C++17。
每題有 3 組測試資料；作答後不需要手動輸入測試資料。


## 作答方式提醒

每題流程：

```text
題目
↓
C 作答 cell ─────→ C 檢查
↓
C++ 作答 cell ───→ C++ 檢查
↓
同一組 3 組測試資料
↓
✅ 通過 / ❌ 顯示差異
```

只修改 `%%c_answer Q...` 與 `%%cpp_answer Q...` cell 裡的程式內容；不要修改檢查 cell。

**重要：** 自動檢查主要驗證輸出正確性；題目要求使用 pointer / array / function pointer 的部分，請自行遵守，才能真正練到該章觀念。


---

# Part A：位址、解參考與基本修改（Q1–Q10）


## Q1. 用指標讀取整數

讀入整數 x，建立 `int *p = &x`，使用 `*p` 輸出 x。

**練習重點：** `&`、`*`、基本解參考

**範例輸入：**

```text
8
```

**範例輸出：**

```text
8
```


### C 作答

In [ ]:
%%c_answer Q1
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q1

In [ ]:
# 檢查 Q1（C）：完成上方 C 程式後執行
_run_check("Q1", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q1
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q1

In [ ]:
# 檢查 Q1（C++）：完成上方 C++ 程式後執行
_run_check("Q1", "CPP")


## Q2. 透過指標加 5

讀入 x，建立指標 p 指向 x，透過 `*p` 將 x 加 5，再輸出 x。

**練習重點：** 透過指標修改變數

**範例輸入：**

```text
10
```

**範例輸出：**

```text
15
```


### C 作答

In [ ]:
%%c_answer Q2
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q2

In [ ]:
# 檢查 Q2（C）：完成上方 C 程式後執行
_run_check("Q2", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q2
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q2

In [ ]:
# 檢查 Q2（C++）：完成上方 C++ 程式後執行
_run_check("Q2", "CPP")


## Q3. 透過指標乘 2

讀入 x，使用指標把 x 乘 2，輸出結果。

**練習重點：** 解參考後寫入

**範例輸入：**

```text
7
```

**範例輸出：**

```text
14
```


### C 作答

In [ ]:
%%c_answer Q3
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q3

In [ ]:
# 檢查 Q3（C）：完成上方 C 程式後執行
_run_check("Q3", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q3
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q3

In [ ]:
# 檢查 Q3（C++）：完成上方 C++ 程式後執行
_run_check("Q3", "CPP")


## Q4. 透過指標改變正負號

讀入 x，使用指標令 `x = -x`，輸出結果。

**練習重點：** 透過 `*p` 修改

**範例輸入：**

```text
12
```

**範例輸出：**

```text
-12
```


### C 作答

In [ ]:
%%c_answer Q4
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q4

In [ ]:
# 檢查 Q4（C）：完成上方 C 程式後執行
_run_check("Q4", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q4
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q4

In [ ]:
# 檢查 Q4（C++）：完成上方 C++ 程式後執行
_run_check("Q4", "CPP")


## Q5. 兩個變數，各用一個指標

讀入 a、b，建立 pa、pb 分別指向它們，輸出 `*pa + *pb`。

**練習重點：** 多個基本指標

**範例輸入：**

```text
3 7
```

**範例輸出：**

```text
10
```


### C 作答

In [ ]:
%%c_answer Q5
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q5

In [ ]:
# 檢查 Q5（C）：完成上方 C 程式後執行
_run_check("Q5", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q5
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q5

In [ ]:
# 檢查 Q5（C++）：完成上方 C++ 程式後執行
_run_check("Q5", "CPP")


## Q6. 用指標計算差

讀入 a、b，透過兩個指標輸出 `*pa - *pb`。

**練習重點：** 讀取多個指標

**範例輸入：**

```text
9 4
```

**範例輸出：**

```text
5
```


### C 作答

In [ ]:
%%c_answer Q6
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q6

In [ ]:
# 檢查 Q6（C）：完成上方 C 程式後執行
_run_check("Q6", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q6
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q6

In [ ]:
# 檢查 Q6（C++）：完成上方 C++ 程式後執行
_run_check("Q6", "CPP")


## Q7. 平方後寫回

讀入 x，透過指標將 x 改成自己的平方，再輸出。

**練習重點：** 讀取與寫入同一位址

**範例輸入：**

```text
5
```

**範例輸出：**

```text
25
```


### C 作答

In [ ]:
%%c_answer Q7
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q7

In [ ]:
# 檢查 Q7（C）：完成上方 C 程式後執行
_run_check("Q7", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q7
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q7

In [ ]:
# 檢查 Q7（C++）：完成上方 C++ 程式後執行
_run_check("Q7", "CPP")


## Q8. 字元指標

讀入一個英文字元 c，建立 `char *p = &c`，輸出 `*p`。

**練習重點：** `char*` 基礎

**範例輸入：**

```text
A
```

**範例輸出：**

```text
A
```


### C 作答

In [ ]:
%%c_answer Q8
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q8

In [ ]:
# 檢查 Q8（C）：完成上方 C 程式後執行
_run_check("Q8", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q8
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q8

In [ ]:
# 檢查 Q8（C++）：完成上方 C++ 程式後執行
_run_check("Q8", "CPP")


## Q9. 字元轉大寫

讀入一個小寫英文字母，透過 `char*` 將它改成大寫並輸出。

**練習重點：** 字元資料的指標修改

**範例輸入：**

```text
a
```

**範例輸出：**

```text
A
```


### C 作答

In [ ]:
%%c_answer Q9
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式
    char c; 
    scanf("%c", &c);
    char *p = &c; 

    if (*p >= 'a' && *p <= 'z') *p = (char)(*p - 'a' + 'A');
    
    printf("%c\n", c);

    return 0;
}


已儲存 Q9 的 C 作答。請執行下方 C 檢查 cell。


### ▶ 執行 C 檢查：Q9

In [6]:
# 檢查 Q9（C）：完成上方 C 程式後執行
_run_check("Q9", "C")


✅ Q9（C）全部通過（3 / 3）


### C++ 作答

In [ ]:
%%cpp_answer Q9
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q9

In [3]:
# 檢查 Q9（C++）：完成上方 C++ 程式後執行
_run_check("Q9", "CPP")


❌ Q9：尚未執行 C++ 作答 cell


## Q10. 安全空指標判斷

建立 `int *p = nullptr`（C 可用 `NULL`）。讀入 x；若 x=0 保持空指標並輸出 `NULL`，否則令 p 指向 x 並輸出 `*p`。

**練習重點：** `nullptr` / `NULL` 與解參考前檢查

**範例輸入：**

```text
0
```

**範例輸出：**

```text
NULL
```


### C 作答

In [ ]:
%%c_answer Q10
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q10

In [ ]:
# 檢查 Q10（C）：完成上方 C 程式後執行
_run_check("Q10", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q10
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q10

In [ ]:
# 檢查 Q10（C++）：完成上方 C++ 程式後執行
_run_check("Q10", "CPP")


---

# Part B：指標作為函式參數（Q11–Q20）


## Q11. 函式 addOne

撰寫 `addOne(int *p)`，讓它把被指向整數加 1。讀入 x、呼叫函式、輸出 x。

**練習重點：** 指標作為函式參數

**範例輸入：**

```text
9
```

**範例輸出：**

```text
10
```


### C 作答

In [ ]:
%%c_answer Q11
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q11

In [ ]:
# 檢查 Q11（C）：完成上方 C 程式後執行
_run_check("Q11", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q11
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q11

In [ ]:
# 檢查 Q11（C++）：完成上方 C++ 程式後執行
_run_check("Q11", "CPP")


## Q12. 函式 setZero

撰寫 `setZero(int *p)` 將被指向值改成 0。

**練習重點：** 用指標參數修改呼叫者資料

**範例輸入：**

```text
35
```

**範例輸出：**

```text
0
```


### C 作答

In [ ]:
%%c_answer Q12
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q12

In [ ]:
# 檢查 Q12（C）：完成上方 C 程式後執行
_run_check("Q12", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q12
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q12

In [ ]:
# 檢查 Q12（C++）：完成上方 C++ 程式後執行
_run_check("Q12", "CPP")


## Q13. 函式 triple

撰寫 `triple(int *p)` 將被指向值乘 3。

**練習重點：** pointer parameter

**範例輸入：**

```text
4
```

**範例輸出：**

```text
12
```


### C 作答

In [ ]:
%%c_answer Q13
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q13

In [ ]:
# 檢查 Q13（C）：完成上方 C 程式後執行
_run_check("Q13", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q13
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q13

In [ ]:
# 檢查 Q13（C++）：完成上方 C++ 程式後執行
_run_check("Q13", "CPP")


## Q14. 用指標交換兩數

撰寫 `swapValues(int *a, int *b)` 交換兩個整數。讀入 a、b，輸出交換後的 `a b`。

**練習重點：** 兩個指標參數

**範例輸入：**

```text
3 8
```

**範例輸出：**

```text
8 3
```


### C 作答

In [ ]:
%%c_answer Q14
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q14

In [ ]:
# 檢查 Q14（C）：完成上方 C 程式後執行
_run_check("Q14", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q14
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q14

In [ ]:
# 檢查 Q14（C++）：完成上方 C++ 程式後執行
_run_check("Q14", "CPP")


## Q15. 由小到大整理兩數

撰寫函式接收兩個 `int*`；若第一個值較大就交換。輸出較小值與較大值。

**練習重點：** 指標參數 + 比較

**範例輸入：**

```text
9 2
```

**範例輸出：**

```text
2 9
```


### C 作答

In [ ]:
%%c_answer Q15
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q15

In [ ]:
# 檢查 Q15（C）：完成上方 C 程式後執行
_run_check("Q15", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q15
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q15

In [ ]:
# 檢查 Q15（C++）：完成上方 C++ 程式後執行
_run_check("Q15", "CPP")


## Q16. 商與餘數雙輸出

讀入 a、b（b 非 0），撰寫函式利用兩個輸出指標回傳整數除法的商與餘數，輸出 `q r`。

**練習重點：** output parameters

**範例輸入：**

```text
17 5
```

**範例輸出：**

```text
3 2
```


### C 作答

In [ ]:
%%c_answer Q16
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q16

In [ ]:
# 檢查 Q16（C）：完成上方 C 程式後執行
_run_check("Q16", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q16
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q16

In [ ]:
# 檢查 Q16（C++）：完成上方 C++ 程式後執行
_run_check("Q16", "CPP")


## Q17. 同時回傳最小與最大

讀入 a、b，撰寫函式透過 `int *mn, int *mx` 回傳兩者最小與最大值。

**練習重點：** 多重輸出指標

**範例輸入：**

```text
8 3
```

**範例輸出：**

```text
3 8
```


### C 作答

In [ ]:
%%c_answer Q17
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q17

In [ ]:
# 檢查 Q17（C）：完成上方 C 程式後執行
_run_check("Q17", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q17
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q17

In [ ]:
# 檢查 Q17（C++）：完成上方 C++ 程式後執行
_run_check("Q17", "CPP")


## Q18. 絕對值函式

撰寫 `makeAbs(int *p)`；若值為負數就改成正數。

**練習重點：** 指標修改 + 條件

**範例輸入：**

```text
-12
```

**範例輸出：**

```text
12
```


### C 作答

In [ ]:
%%c_answer Q18
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q18

In [ ]:
# 檢查 Q18（C）：完成上方 C 程式後執行
_run_check("Q18", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q18
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q18

In [ ]:
# 檢查 Q18（C++）：完成上方 C++ 程式後執行
_run_check("Q18", "CPP")


## Q19. 字元轉小寫函式

讀入一個大寫英文字母，撰寫函式以 `char*` 將其轉成小寫。

**練習重點：** `char*` 函式參數

**範例輸入：**

```text
A
```

**範例輸出：**

```text
a
```


### C 作答

In [ ]:
%%c_answer Q19
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q19

In [ ]:
# 檢查 Q19（C）：完成上方 C 程式後執行
_run_check("Q19", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q19
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q19

In [ ]:
# 檢查 Q19（C++）：完成上方 C++ 程式後執行
_run_check("Q19", "CPP")


## Q20. 兩個值同時加上 k

讀入 a、b、k，撰寫函式接收 `int *a, int *b, int k`，兩個值都加 k，輸出結果。

**練習重點：** 多個指標與一般參數

**範例輸入：**

```text
2 5 3
```

**範例輸出：**

```text
5 8
```


### C 作答

In [ ]:
%%c_answer Q20
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q20

In [ ]:
# 檢查 Q20（C）：完成上方 C 程式後執行
_run_check("Q20", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q20
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q20

In [ ]:
# 檢查 Q20（C++）：完成上方 C++ 程式後執行
_run_check("Q20", "CPP")


---

# Part C：指向、別名與回傳指標（Q21–Q30）


## Q21. 兩個指標指向同一變數

讀入 x，令 p 與 q 都指向 x；先用 p 加 2，再用 q 乘 3，輸出 x。

**練習重點：** aliasing 基礎

**範例輸入：**

```text
4
```

**範例輸出：**

```text
18
```


### C 作答

In [ ]:
%%c_answer Q21
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q21

In [ ]:
# 檢查 Q21（C）：完成上方 C 程式後執行
_run_check("Q21", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q21
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q21

In [ ]:
# 檢查 Q21（C++）：完成上方 C++ 程式後執行
_run_check("Q21", "CPP")


## Q22. 複製指標

讀入 x，令 `p=&x`、`q=p`，透過 q 將值加 10，再透過 p 輸出。

**練習重點：** pointer copy

**範例輸入：**

```text
5
```

**範例輸出：**

```text
15
```


### C 作答

In [ ]:
%%c_answer Q22
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q22

In [ ]:
# 檢查 Q22（C）：完成上方 C 程式後執行
_run_check("Q22", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q22
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q22

In [ ]:
# 檢查 Q22（C++）：完成上方 C++ 程式後執行
_run_check("Q22", "CPP")


## Q23. 重新指定指標

讀入 a、b、selector。若 selector=0 讓 p 指向 a，否則指向 b；輸出 `*p`。

**練習重點：** pointer reassignment

**範例輸入：**

```text
3 8 0
```

**範例輸出：**

```text
3
```


### C 作答

In [ ]:
%%c_answer Q23
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q23

In [ ]:
# 檢查 Q23（C）：完成上方 C 程式後執行
_run_check("Q23", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q23
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q23

In [ ]:
# 檢查 Q23（C++）：完成上方 C++ 程式後執行
_run_check("Q23", "CPP")


## Q24. 選擇較大者的指標

讀入 a、b，令 p 指向較大的變數；若相等指向 a。輸出 `*p`。

**練習重點：** 條件式改變指標指向

**範例輸入：**

```text
3 9
```

**範例輸出：**

```text
9
```


### C 作答

In [ ]:
%%c_answer Q24
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q24

In [ ]:
# 檢查 Q24（C）：完成上方 C 程式後執行
_run_check("Q24", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q24
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q24

In [ ]:
# 檢查 Q24（C++）：完成上方 C++ 程式後執行
_run_check("Q24", "CPP")


## Q25. 選擇較小者並修改

讀入 a、b，令 p 指向較小者；若相等指向 a。透過 p 將較小值加 100，輸出 `a b`。

**練習重點：** 選擇位址後修改

**範例輸入：**

```text
3 9
```

**範例輸出：**

```text
103 9
```


### C 作答

In [ ]:
%%c_answer Q25
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q25

In [ ]:
# 檢查 Q25（C）：完成上方 C 程式後執行
_run_check("Q25", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q25
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q25

In [ ]:
# 檢查 Q25（C++）：完成上方 C++ 程式後執行
_run_check("Q25", "CPP")


## Q26. const 指標讀取

讀入 x，使用 `const int *p = &x` 讀取並輸出 x 的兩倍。

**練習重點：** pointer to const 的讀取概念

**範例輸入：**

```text
6
```

**範例輸出：**

```text
12
```


### C 作答

In [ ]:
%%c_answer Q26
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q26

In [ ]:
# 檢查 Q26（C）：完成上方 C 程式後執行
_run_check("Q26", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q26
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q26

In [ ]:
# 檢查 Q26（C++）：完成上方 C++ 程式後執行
_run_check("Q26", "CPP")


## Q27. 指標本身固定的練習

讀入 x，建立指向 x 的固定指標（C++ 可用 `int *const p`；C 可用一般指標模擬），透過它將 x 加 7，輸出。

**練習重點：** pointer constness 概念

**範例輸入：**

```text
1
```

**範例輸出：**

```text
8
```


### C 作答

In [ ]:
%%c_answer Q27
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q27

In [ ]:
# 檢查 Q27（C）：完成上方 C 程式後執行
_run_check("Q27", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q27
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q27

In [ ]:
# 檢查 Q27（C++）：完成上方 C++ 程式後執行
_run_check("Q27", "CPP")


## Q28. 回傳較大值的位址

撰寫函式接收兩個 `int*` 並回傳指向較大值的 `int*`。讀入 a、b，輸出回傳指標所指的值。

**練習重點：** 回傳指標

**範例輸入：**

```text
4 11
```

**範例輸出：**

```text
11
```


### C 作答

In [ ]:
%%c_answer Q28
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q28

In [ ]:
# 檢查 Q28（C）：完成上方 C 程式後執行
_run_check("Q28", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q28
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q28

In [ ]:
# 檢查 Q28（C++）：完成上方 C++ 程式後執行
_run_check("Q28", "CPP")


## Q29. 回傳較小值的位址

撰寫函式接收兩個 `int*` 並回傳指向較小值的 `int*`。

**練習重點：** 函式回傳 pointer

**範例輸入：**

```text
4 11
```

**範例輸出：**

```text
4
```


### C 作答

In [ ]:
%%c_answer Q29
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q29

In [ ]:
# 檢查 Q29（C）：完成上方 C 程式後執行
_run_check("Q29", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q29
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q29

In [ ]:
# 檢查 Q29（C++）：完成上方 C++ 程式後執行
_run_check("Q29", "CPP")


## Q30. 兩個 pointer parameter 做運算

讀入 a、b，撰寫 `sum(const int *a, const int *b)` 回傳兩者和。

**練習重點：** 唯讀 pointer parameter

**範例輸入：**

```text
10 20
```

**範例輸出：**

```text
30
```


### C 作答

In [ ]:
%%c_answer Q30
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q30

In [ ]:
# 檢查 Q30（C）：完成上方 C 程式後執行
_run_check("Q30", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q30
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q30

In [ ]:
# 檢查 Q30（C++）：完成上方 C++ 程式後執行
_run_check("Q30", "CPP")


---

# Part D：常見錯誤與安全寫法（Q31–Q40）


## Q31. 修正：未初始化指標

寫出安全程式：讀入 x，建立一個有效指標指向 x，將值加 1 後輸出。不要讓指標在未指向有效物件時被解參考。

**練習重點：** 避免未初始化指標

**範例輸入：**

```text
8
```

**範例輸出：**

```text
9
```


### C 作答

In [ ]:
%%c_answer Q31
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q31

In [ ]:
# 檢查 Q31（C）：完成上方 C 程式後執行
_run_check("Q31", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q31
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q31

In [ ]:
# 檢查 Q31（C++）：完成上方 C++ 程式後執行
_run_check("Q31", "CPP")


## Q32. 修正：值不是位址

讀入 x，正確地讓 `int*` 指向 x，而不是把 x 的數值當位址；輸出 `*p`。

**練習重點：** `p=&x` 而不是 `p=x`

**範例輸入：**

```text
14
```

**範例輸出：**

```text
14
```


### C 作答

In [ ]:
%%c_answer Q32
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q32

In [ ]:
# 檢查 Q32（C）：完成上方 C 程式後執行
_run_check("Q32", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q32
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q32

In [ ]:
# 檢查 Q32（C++）：完成上方 C++ 程式後執行
_run_check("Q32", "CPP")


## Q33. 修正：缺少解參考

讀入 x，請輸出指標所指的值，而不是輸出位址。

**練習重點：** 區分 `p` 與 `*p`

**範例輸入：**

```text
99
```

**範例輸出：**

```text
99
```


### C 作答

In [ ]:
%%c_answer Q33
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q33

In [ ]:
# 檢查 Q33（C）：完成上方 C 程式後執行
_run_check("Q33", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q33
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q33

In [ ]:
# 檢查 Q33（C++）：完成上方 C++ 程式後執行
_run_check("Q33", "CPP")


## Q34. 修正：錯誤型態

讀入 double x，使用 `double*` 指向它，將值乘 2，固定輸出到小數點後 2 位。

**練習重點：** 指標型態需相容

**範例輸入：**

```text
1.5
```

**範例輸出：**

```text
3.00
```


### C 作答

In [ ]:
%%c_answer Q34
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q34

In [ ]:
# 檢查 Q34（C）：完成上方 C 程式後執行
_run_check("Q34", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q34
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q34

In [ ]:
# 檢查 Q34（C++）：完成上方 C++ 程式後執行
_run_check("Q34", "CPP")


## Q35. 修正：空指標不可解參考

讀入 flag 與 x。flag=1 時讓 p 指向 x 並輸出值；flag=0 時輸出 `NULL`，不可解參考空指標。

**練習重點：** null check

**範例輸入：**

```text
1 42
```

**範例輸出：**

```text
42
```


### C 作答

In [ ]:
%%c_answer Q35
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q35

In [ ]:
# 檢查 Q35（C）：完成上方 C 程式後執行
_run_check("Q35", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q35
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q35

In [ ]:
# 檢查 Q35（C++）：完成上方 C++ 程式後執行
_run_check("Q35", "CPP")


## Q36. 修正 swap 函式

完成一個真正能改變呼叫者變數的 swap 函式，必須使用兩個指標參數。

**練習重點：** pass-by-pointer

**範例輸入：**

```text
1 2
```

**範例輸出：**

```text
2 1
```


### C 作答

In [ ]:
%%c_answer Q36
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q36

In [ ]:
# 檢查 Q36（C）：完成上方 C 程式後執行
_run_check("Q36", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q36
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q36

In [ ]:
# 檢查 Q36（C++）：完成上方 C++ 程式後執行
_run_check("Q36", "CPP")


## Q37. 修正 increment 函式

避免只修改參數的副本：使用 `int*` 讓函式真的把呼叫者的 x 加 1。

**練習重點：** 指標與值傳遞差異

**範例輸入：**

```text
5
```

**範例輸出：**

```text
6
```


### C 作答

In [ ]:
%%c_answer Q37
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q37

In [ ]:
# 檢查 Q37（C）：完成上方 C 程式後執行
_run_check("Q37", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q37
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q37

In [ ]:
# 檢查 Q37（C++）：完成上方 C++ 程式後執行
_run_check("Q37", "CPP")


## Q38. 先改指向，再改值

讀入 a、b。p 一開始指向 a；接著重新令 p 指向 b，透過 p 把 b 乘 2；輸出 `a b`。

**練習重點：** 重新指定 pointer 不會改原先變數

**範例輸入：**

```text
3 4
```

**範例輸出：**

```text
3 8
```


### C 作答

In [ ]:
%%c_answer Q38
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q38

In [ ]:
# 檢查 Q38（C）：完成上方 C 程式後執行
_run_check("Q38", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q38
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q38

In [ ]:
# 檢查 Q38（C++）：完成上方 C++ 程式後執行
_run_check("Q38", "CPP")


## Q39. 字元指標安全修改

讀入一個英文字母。若為小寫，用 `char*` 轉成大寫；否則保持不變。

**練習重點：** `char*` + 條件

**範例輸入：**

```text
b
```

**範例輸出：**

```text
B
```


### C 作答

In [ ]:
%%c_answer Q39
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q39

In [ ]:
# 檢查 Q39（C）：完成上方 C 程式後執行
_run_check("Q39", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q39
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q39

In [ ]:
# 檢查 Q39（C++）：完成上方 C++ 程式後執行
_run_check("Q39", "CPP")


## Q40. 雙重別名更新

讀入 x，建立 p、q 都指向 x。用 p 加 3，再用 q 減 1，輸出 x。

**練習重點：** aliasing 與共享資料

**範例輸入：**

```text
10
```

**範例輸出：**

```text
12
```


### C 作答

In [ ]:
%%c_answer Q40
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q40

In [ ]:
# 檢查 Q40（C）：完成上方 C 程式後執行
_run_check("Q40", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q40
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q40

In [ ]:
# 檢查 Q40（C++）：完成上方 C++ 程式後執行
_run_check("Q40", "CPP")


---

# Part E：綜合練習（Q41–Q50）


## Q41. normalizePair

讀入 a、b。使用指標函式：先把兩數都取絕對值，再確保 a<=b，輸出 `a b`。

**練習重點：** 整合：修改 + swap

**範例輸入：**

```text
-8 3
```

**範例輸出：**

```text
3 8
```


### C 作答

In [ ]:
%%c_answer Q41
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q41

In [ ]:
# 檢查 Q41（C）：完成上方 C 程式後執行
_run_check("Q41", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q41
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q41

In [ ]:
# 檢查 Q41（C++）：完成上方 C++ 程式後執行
_run_check("Q41", "CPP")


## Q42. scaleAndShift

讀入 x、scale、shift。函式接收 `int*`，先乘 scale 再加 shift，輸出結果。

**練習重點：** 整合 pointer parameter

**範例輸入：**

```text
3 4 2
```

**範例輸出：**

```text
14
```


### C 作答

In [ ]:
%%c_answer Q42
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q42

In [ ]:
# 檢查 Q42（C）：完成上方 C 程式後執行
_run_check("Q42", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q42
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q42

In [ ]:
# 檢查 Q42（C++）：完成上方 C++ 程式後執行
_run_check("Q42", "CPP")


## Q43. 兩數距離

讀入 a、b，使用唯讀指標參數函式回傳 `|a-b|`。

**練習重點：** const pointer parameter

**範例輸入：**

```text
10 3
```

**範例輸出：**

```text
7
```


### C 作答

In [ ]:
%%c_answer Q43
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q43

In [ ]:
# 檢查 Q43（C）：完成上方 C 程式後執行
_run_check("Q43", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q43
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q43

In [ ]:
# 檢查 Q43（C++）：完成上方 C++ 程式後執行
_run_check("Q43", "CPP")


## Q44. 更新較小值

讀入 a、b、k，找出較小值的位址並只替它加 k；相等時修改 a。輸出 `a b`。

**練習重點：** 回傳/選擇 pointer + 修改

**範例輸入：**

```text
2 8 5
```

**範例輸出：**

```text
7 8
```


### C 作答

In [ ]:
%%c_answer Q44
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q44

In [ ]:
# 檢查 Q44（C）：完成上方 C 程式後執行
_run_check("Q44", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q44
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q44

In [ ]:
# 檢查 Q44（C++）：完成上方 C++ 程式後執行
_run_check("Q44", "CPP")


## Q45. 更新較大值

讀入 a、b、k，找出較大值的位址並只替它減 k；相等時修改 a。

**練習重點：** pointer selection

**範例輸入：**

```text
2 8 3
```

**範例輸出：**

```text
2 5
```


### C 作答

In [ ]:
%%c_answer Q45
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q45

In [ ]:
# 檢查 Q45（C）：完成上方 C 程式後執行
_run_check("Q45", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q45
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q45

In [ ]:
# 檢查 Q45（C++）：完成上方 C++ 程式後執行
_run_check("Q45", "CPP")


## Q46. 四則結果用輸出指標回傳

讀入 a、b。函式透過四個 `int*` 回傳 sum、difference、product、整數 quotient（假設 b 非 0），依序輸出。

**練習重點：** 多重 output parameters

**範例輸入：**

```text
8 2
```

**範例輸出：**

```text
10 6 16 4
```


### C 作答

In [ ]:
%%c_answer Q46
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q46

In [ ]:
# 檢查 Q46（C）：完成上方 C 程式後執行
_run_check("Q46", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q46
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q46

In [ ]:
# 檢查 Q46（C++）：完成上方 C++ 程式後執行
_run_check("Q46", "CPP")


## Q47. 夾限 clamp

讀入 x、low、high（low<=high），函式用 `int*` 把 x 限制在 [low, high]，輸出 x。

**練習重點：** pointer mutation + range

**範例輸入：**

```text
15 0 10
```

**範例輸出：**

```text
10
```


### C 作答

In [ ]:
%%c_answer Q47
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q47

In [ ]:
# 檢查 Q47（C）：完成上方 C 程式後執行
_run_check("Q47", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q47
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q47

In [ ]:
# 檢查 Q47（C++）：完成上方 C++ 程式後執行
_run_check("Q47", "CPP")


## Q48. 有條件交換

讀入 a、b、flag。flag=1 時用 pointer swap，flag=0 時不交換。輸出 `a b`。

**練習重點：** pointer function + control

**範例輸入：**

```text
1 9 1
```

**範例輸出：**

```text
9 1
```


### C 作答

In [ ]:
%%c_answer Q48
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q48

In [ ]:
# 檢查 Q48（C）：完成上方 C 程式後執行
_run_check("Q48", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q48
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q48

In [ ]:
# 檢查 Q48（C++）：完成上方 C++ 程式後執行
_run_check("Q48", "CPP")


## Q49. 兩階段更新

讀入 x。先呼叫 pointer 函式加 5，再呼叫另一個 pointer 函式乘 2，輸出結果。

**練習重點：** 多個 pointer function 串接

**範例輸入：**

```text
3
```

**範例輸出：**

```text
16
```


### C 作答

In [ ]:
%%c_answer Q49
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q49

In [ ]:
# 檢查 Q49（C）：完成上方 C 程式後執行
_run_check("Q49", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q49
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q49

In [ ]:
# 檢查 Q49（C++）：完成上方 C++ 程式後執行
_run_check("Q49", "CPP")


## Q50. 基本指標總整合

讀入 a、b。建立 pa、pb；透過指標把 a 加 1、b 減 1，再用 pointer swap，最後輸出 `a b`。

**練習重點：** 基本 pointer 綜合

**範例輸入：**

```text
3 8
```

**範例輸出：**

```text
7 4
```


### C 作答

In [ ]:
%%c_answer Q50
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>

int main(void) {
    // TODO: 請在這裡完成 C 程式

    return 0;
}


### ▶ 執行 C 檢查：Q50

In [ ]:
# 檢查 Q50（C）：完成上方 C 程式後執行
_run_check("Q50", "C")


### C++ 作答

In [ ]:
%%cpp_answer Q50
#include <iostream>
#include <string>
#include <iomanip>
#include <algorithm>
#include <cctype>
using namespace std;

int main() {
    // TODO: 請在這裡完成 C++ 程式

    return 0;
}


### ▶ 執行 C++ 檢查：Q50

In [ ]:
# 檢查 Q50（C++）：完成上方 C++ 程式後執行
_run_check("Q50", "CPP")


---

# 總檢查


In [ ]:
_final_report()
